# ⚡ SmartPrice Engine — EDA + Training (v2)
**Dataset:** Dynamic Pricing Dataset — [Kaggle](https://www.kaggle.com/datasets/arashnic/dynamic-pricing-dataset)  
**Goal:** Predict optimal ride price with maximum accuracy using feature engineering, log-target transformation, and hyperparameter tuning.

---
### Sections
1. Setup & Load
2. Target Distribution & Log Transform
3. Categorical Features vs Price
4. Numerical Features vs Price
5. Feature Engineering (Interactions + Flags)
6. Correlation Heatmap
7. Baseline XGBoost
8. Optuna Hyperparameter Tuning
9. Final Model — XGBoost vs LightGBM
10. Feature Importance
11. Prediction Error Analysis
12. Save Artifacts

In [ ]:
# ─── Install if needed ───────────────────────────────────────────────────────
import subprocess, sys
def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['optuna', 'lightgbm', 'xgboost', 'scikit-learn', 'joblib', 'seaborn', 'matplotlib']:
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        install(pkg)

print('All packages ready ✅')

In [ ]:
# ─── 1. Setup & Load ─────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings, os, joblib, json
warnings.filterwarnings('ignore')
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# ── Dark theme ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#0f1117',
    'axes.edgecolor':   '#2a2d3a',
    'axes.labelcolor':  '#9ca3af',
    'xtick.color':      '#9ca3af',
    'ytick.color':      '#9ca3af',
    'text.color':       '#e8eaf0',
    'grid.color':       '#1e2130',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'font.family':      'monospace',
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
    'axes.titlepad':    12,
    'figure.dpi':       120,
})

ACCENT  = '#00e5a0'
ACCENT2 = '#005eff'
WARN    = '#ff6b35'
PURPLE  = '#a78bfa'
YELLOW  = '#f59e0b'
PALETTE = [ACCENT, ACCENT2, WARN, PURPLE, YELLOW]

# ── Load data ─────────────────────────────────────────────────────────────────
DATA_PATH = '../data/dynamic_pricing.csv'
df = pd.read_csv(DATA_PATH)

print(f'Shape      : {df.shape}')
print(f'Duplicates : {df.duplicated().sum()}')
print(f'Nulls      : {df.isnull().sum().sum()}')
print(f'Columns    : {list(df.columns)}')
df.head()

In [ ]:
df.describe().round(2)

In [ ]:
# ─── 2. Target Distribution & Log Transform ───────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Target Variable — Price Distribution & Log Transform', fontsize=14, fontweight='bold')

price      = df['Historical_Cost_of_Ride']
log_price  = np.log1p(price)

# Raw histogram
axes[0,0].hist(price, bins=40, color=ACCENT, alpha=0.85, edgecolor='#0f1117')
axes[0,0].axvline(price.mean(),   color=WARN,   linestyle='--', lw=1.5, label=f'Mean   ${price.mean():.0f}')
axes[0,0].axvline(price.median(), color=ACCENT2, linestyle='--', lw=1.5, label=f'Median ${price.median():.0f}')
axes[0,0].set_title('Raw Price — Histogram')
axes[0,0].set_xlabel('Price ($)')
axes[0,0].legend(framealpha=0.2)
axes[0,0].grid(True)

# Raw box
axes[0,1].boxplot(price, patch_artist=True,
    boxprops=dict(facecolor=ACCENT, alpha=0.4),
    medianprops=dict(color=WARN, linewidth=2),
    whiskerprops=dict(color='#9ca3af'),
    capprops=dict(color='#9ca3af'),
    flierprops=dict(marker='o', color=ACCENT2, markersize=3, alpha=0.5))
axes[0,1].set_title('Raw Price — Box Plot')
axes[0,1].set_ylabel('Price ($)')
axes[0,1].grid(True)

# Log histogram
axes[1,0].hist(log_price, bins=40, color=PURPLE, alpha=0.85, edgecolor='#0f1117')
axes[1,0].axvline(log_price.mean(),   color=WARN,   linestyle='--', lw=1.5, label=f'Mean  {log_price.mean():.2f}')
axes[1,0].axvline(log_price.median(), color=ACCENT2, linestyle='--', lw=1.5, label=f'Median {log_price.median():.2f}')
axes[1,0].set_title('log1p(Price) — Much More Normal ✅')
axes[1,0].set_xlabel('log1p(Price)')
axes[1,0].legend(framealpha=0.2)
axes[1,0].grid(True)

# Q-Q comparison
from scipy import stats
(osm_raw, osr_raw), _ = stats.probplot(price,      dist='norm')
(osm_log, osr_log), _ = stats.probplot(log_price,  dist='norm')
axes[1,1].scatter(osm_raw, osr_raw, s=5, alpha=0.4, color=WARN,   label='Raw price')
axes[1,1].scatter(osm_log, osr_log, s=5, alpha=0.4, color=ACCENT, label='log1p price')
lims = [min(osm_raw[0], osm_log[0]), max(osm_raw[-1], osm_log[-1])]
axes[1,1].plot(lims, lims, 'w--', lw=1, alpha=0.3)
axes[1,1].set_title('Q-Q Plot — Raw vs Log Transform')
axes[1,1].set_xlabel('Theoretical Quantiles')
axes[1,1].set_ylabel('Sample Quantiles')
axes[1,1].legend(framealpha=0.2)
axes[1,1].grid(True)

plt.tight_layout()
plt.show()

print(f'Raw  Skewness: {price.skew():.3f}')
print(f'Log  Skewness: {log_price.skew():.3f}  ← closer to 0 = better for tree models')

In [ ]:
# ─── 3. Categorical Features vs Price ─────────────────────────────────────────
cat_cols = ['Vehicle_Type', 'Customer_Loyalty_Status', 'Location_Category', 'Time_of_Booking']

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Categorical Features vs Price', fontsize=14, fontweight='bold')
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    order = df.groupby(col)['Historical_Cost_of_Ride'].median().sort_values(ascending=False).index
    n     = df[col].nunique()
    sns.boxplot(
        data=df, x=col, y='Historical_Cost_of_Ride',
        order=order, hue=col,
        palette=PALETTE[:n],
        legend=False, ax=axes[i]
    )
    for j, cat in enumerate(order):
        med = df[df[col] == cat]['Historical_Cost_of_Ride'].median()
        mn  = df[df[col] == cat]['Historical_Cost_of_Ride'].mean()
        axes[i].text(j, med + 15, f'${med:.0f}', ha='center', fontsize=8, color=ACCENT)
        axes[i].text(j, med - 40, f'μ${mn:.0f}', ha='center', fontsize=7, color='#9ca3af')
    axes[i].set_title(col.replace('_', ' '))
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Price ($)' if i % 2 == 0 else '')
    axes[i].grid(True, axis='y')

plt.tight_layout()
plt.show()

# Summary table
for col in cat_cols:
    print(f'\n── {col} ──')
    summary = df.groupby(col)['Historical_Cost_of_Ride'].agg(['mean','median','std','count']).round(2)
    summary.columns = ['Mean $', 'Median $', 'Std $', 'Count']
    print(summary.sort_values('Median $', ascending=False).to_string())

In [ ]:
# ─── 4. Numerical Features vs Price ───────────────────────────────────────────
num_cols = ['Number_of_Riders', 'Number_of_Drivers', 'Number_of_Past_Rides',
            'Average_Ratings', 'Expected_Ride_Duration']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('Numerical Features vs Price', fontsize=14, fontweight='bold')
axes = axes.flatten()

for i, col in enumerate(num_cols):
    corr   = df[col].corr(df['Historical_Cost_of_Ride'])
    color  = ACCENT if corr >= 0 else WARN
    axes[i].scatter(df[col], df['Historical_Cost_of_Ride'],
                    alpha=0.25, s=8, color=color)
    z      = np.polyfit(df[col], df['Historical_Cost_of_Ride'], 1)
    x_line = np.linspace(df[col].min(), df[col].max(), 200)
    axes[i].plot(x_line, np.poly1d(z)(x_line),
                 color=ACCENT2, linewidth=1.8, linestyle='--')
    strength = 'Strong' if abs(corr) > 0.5 else 'Moderate' if abs(corr) > 0.3 else 'Weak'
    axes[i].set_title(f'{col.replace("_"," ")}\nr = {corr:.3f}  ({strength})')
    axes[i].set_xlabel(col.replace('_', ' '))
    axes[i].set_ylabel('Price ($)' if i % 3 == 0 else '')
    axes[i].grid(True)

axes[-1].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# ─── 5. Feature Engineering ───────────────────────────────────────────────────
df_fe = df.copy()

# Core interactions
df_fe['demand_supply_ratio']  = df_fe['Number_of_Riders'] / (df_fe['Number_of_Drivers'] + 1)
df_fe['riders_per_duration']  = df_fe['Number_of_Riders'] / (df_fe['Expected_Ride_Duration'] + 1)
df_fe['drivers_per_duration'] = df_fe['Number_of_Drivers'] / (df_fe['Expected_Ride_Duration'] + 1)

# Loyalty encoding
loyalty_map = {'Silver': 0, 'Regular': 1, 'Gold': 2}
df_fe['loyalty_score'] = df_fe['Customer_Loyalty_Status'].map(loyalty_map).fillna(1)

# Rating × loyalty interaction
df_fe['rating_loyalty']  = df_fe['Average_Ratings'] * df_fe['loyalty_score']
df_fe['rating_duration'] = df_fe['Average_Ratings'] * df_fe['Expected_Ride_Duration']

# Demand pressure flag
df_fe['high_demand']    = (df_fe['demand_supply_ratio'] > df_fe['demand_supply_ratio'].quantile(0.75)).astype(int)
df_fe['peak_high_demand'] = ((df_fe['high_demand'] == 1) &
                              (df_fe['Time_of_Booking'].isin(['Evening','Night']))).astype(int)

# Experience proxy
df_fe['experience_score'] = df_fe['Number_of_Past_Rides'] * df_fe['Average_Ratings']

engineered = ['demand_supply_ratio','riders_per_duration','drivers_per_duration',
              'rating_loyalty','rating_duration','high_demand','peak_high_demand','experience_score']

# Plot new features correlation
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Engineered Features vs Price', fontsize=14, fontweight='bold')
axes = axes.flatten()

for i, col in enumerate(engineered):
    corr   = df_fe[col].corr(df_fe['Historical_Cost_of_Ride'])
    color  = ACCENT if corr >= 0 else WARN
    if df_fe[col].nunique() <= 2:
        medians = df_fe.groupby(col)['Historical_Cost_of_Ride'].median()
        axes[i].bar(medians.index.astype(str), medians.values, color=[ACCENT, ACCENT2], alpha=0.8)
        axes[i].set_xlabel(col.replace('_',' '))
    else:
        axes[i].scatter(df_fe[col], df_fe['Historical_Cost_of_Ride'],
                        alpha=0.25, s=8, color=color)
        z = np.polyfit(df_fe[col], df_fe['Historical_Cost_of_Ride'], 1)
        x_line = np.linspace(df_fe[col].min(), df_fe[col].max(), 200)
        axes[i].plot(x_line, np.poly1d(z)(x_line), color=ACCENT2, lw=1.5, ls='--')
    axes[i].set_title(f'{col.replace("_"," ")}\nr = {corr:.3f}')
    axes[i].set_ylabel('Price ($)' if i % 4 == 0 else '')
    axes[i].grid(True)

plt.tight_layout()
plt.show()

print('\nNew features correlation with price:')
for col in engineered:
    corr = df_fe[col].corr(df_fe['Historical_Cost_of_Ride'])
    bar  = '█' * int(abs(corr) * 25)
    sign = '+' if corr >= 0 else '-'
    print(f'  {sign} {col:<30} {bar} {corr:.3f}')

In [ ]:
# ─── 6. Correlation Heatmap ───────────────────────────────────────────────────
df_enc = df_fe.copy()
for col in ['Location_Category','Customer_Loyalty_Status','Time_of_Booking','Vehicle_Type']:
    df_enc[col] = df_enc[col].astype('category').cat.codes

# Only keep numeric
df_num = df_enc.select_dtypes(include=[np.number])
corr_matrix = df_num.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(14, 11))
cmap = sns.diverging_palette(220, 150, s=80, l=55, as_cmap=True)
sns.heatmap(
    corr_matrix, mask=mask, cmap=cmap,
    annot=True, fmt='.2f', annot_kws={'size': 7},
    center=0, vmax=1, vmin=-1,
    square=True, linewidths=0.3, linecolor='#1a1d26',
    cbar_kws={'shrink': 0.65}, ax=ax
)
ax.set_title('Feature Correlation Matrix (All Features)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nTop correlations with price (all features):')
target_corr = corr_matrix['Historical_Cost_of_Ride'].drop('Historical_Cost_of_Ride').abs().sort_values(ascending=False)
for feat, val in target_corr.head(12).items():
    bar = '█' * int(val * 25)
    print(f'  {feat:<32} {bar} {val:.3f}')

In [ ]:
# ─── 7. Prepare Final Feature Matrix ──────────────────────────────────────────
df_model = df_fe.copy()

# Encode categoricals
df_model['loyalty_encoded']  = df_model['Customer_Loyalty_Status'].map({'Silver':0,'Regular':1,'Gold':2}).fillna(1)
df_model['location_encoded'] = df_model['Location_Category'].map({'Rural':0,'Suburban':1,'Urban':2}).fillna(1)
df_model['time_encoded']     = df_model['Time_of_Booking'].map({'Morning':0,'Afternoon':1,'Evening':2,'Night':3}).fillna(1)
le = LabelEncoder()
df_model['vehicle_encoded']  = le.fit_transform(df_model['Vehicle_Type'])

FEATURES = [
    # Original
    'Number_of_Riders', 'Number_of_Drivers',
    'Number_of_Past_Rides', 'Average_Ratings', 'Expected_Ride_Duration',
    'loyalty_encoded', 'location_encoded', 'vehicle_encoded', 'time_encoded',
    # Engineered
    'demand_supply_ratio', 'riders_per_duration', 'drivers_per_duration',
    'rating_loyalty', 'rating_duration',
    'high_demand', 'peak_high_demand', 'experience_score',
]

X = df_model[FEATURES]
y = df_model['Historical_Cost_of_Ride']
y_log = np.log1p(y)   # ← log transform target

X_train, X_test, y_train, y_test = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)
y_test_orig  = np.expm1(y_test)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train size : {X_train_sc.shape}')
print(f'Test size  : {X_test_sc.shape}')
print(f'Features   : {len(FEATURES)}')
print(f'Target     : log1p(price) — will inverse-transform for evaluation ✅')

In [ ]:
# ─── 8. Baseline XGBoost (before tuning) ──────────────────────────────────────
def evaluate(model, X_tr, y_tr, X_te, y_te_log, label='Model'):
    """Fit, predict, and return metrics — handles log-space target."""
    model.fit(X_tr, y_tr)
    y_pred_log  = model.predict(X_te)
    y_pred      = np.expm1(y_pred_log)
    y_actual    = np.expm1(y_te_log)
    mae   = mean_absolute_error(y_actual, y_pred)
    rmse  = np.sqrt(mean_squared_error(y_actual, y_pred))
    r2    = r2_score(y_actual, y_pred)
    mape  = np.mean(np.abs((y_actual - y_pred) / y_actual)) * 100
    cv    = cross_val_score(model, scaler.transform(X), y_log, cv=5, scoring='r2')
    print(f'\n── {label} ──────────────────────────────')
    print(f'  MAE   : ${mae:.2f}')
    print(f'  RMSE  : ${rmse:.2f}')
    print(f'  R²    : {r2:.4f}')
    print(f'  MAPE  : {mape:.2f}%')
    print(f'  CV R² : {cv.mean():.4f} ± {cv.std():.4f}')
    print('─' * 42)
    return {'mae': mae, 'rmse': rmse, 'r2': r2, 'mape': mape,
            'cv_mean': cv.mean(), 'cv_std': cv.std(),
            'y_pred': y_pred, 'y_actual': y_actual}

baseline = XGBRegressor(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbosity=0
)
base_results = evaluate(baseline, X_train_sc, y_train, X_test_sc, y_test, 'Baseline XGBoost')

In [ ]:
# ─── 9. Optuna Hyperparameter Tuning ──────────────────────────────────────────
def objective_xgb(trial):
    params = dict(
        n_estimators     = trial.suggest_int('n_estimators', 200, 800),
        max_depth        = trial.suggest_int('max_depth', 3, 6),
        learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        subsample        = trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0),
        min_child_weight = trial.suggest_int('min_child_weight', 1, 10),
        gamma            = trial.suggest_float('gamma', 0, 0.5),
        reg_alpha        = trial.suggest_float('reg_alpha', 0, 1.0),
        reg_lambda       = trial.suggest_float('reg_lambda', 0.5, 3.0),
        random_state=42, n_jobs=-1, verbosity=0
    )
    model = XGBRegressor(**params)
    cv    = cross_val_score(model, X_train_sc, y_train, cv=5, scoring='r2')
    return cv.mean()

def objective_lgbm(trial):
    params = dict(
        n_estimators     = trial.suggest_int('n_estimators', 200, 800),
        max_depth        = trial.suggest_int('max_depth', 3, 7),
        learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        num_leaves       = trial.suggest_int('num_leaves', 20, 80),
        subsample        = trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0),
        min_child_samples= trial.suggest_int('min_child_samples', 5, 30),
        reg_alpha        = trial.suggest_float('reg_alpha', 0, 1.0),
        reg_lambda       = trial.suggest_float('reg_lambda', 0.5, 3.0),
        random_state=42, n_jobs=-1, verbose=-1
    )
    model = LGBMRegressor(**params)
    cv    = cross_val_score(model, X_train_sc, y_train, cv=5, scoring='r2')
    return cv.mean()

print('🔍 Tuning XGBoost (50 trials)...')
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=50, show_progress_bar=False)
print(f'   Best XGB CV R²: {study_xgb.best_value:.4f}')
print(f'   Best params: {study_xgb.best_params}')

print('\n🔍 Tuning LightGBM (50 trials)...')
study_lgbm = optuna.create_study(direction='maximize')
study_lgbm.optimize(objective_lgbm, n_trials=50, show_progress_bar=False)
print(f'   Best LGBM CV R²: {study_lgbm.best_value:.4f}')
print(f'   Best params: {study_lgbm.best_params}')

In [ ]:
# ─── 10. Final Models — XGBoost vs LightGBM ───────────────────────────────────
final_xgb = XGBRegressor(
    **study_xgb.best_params,
    random_state=42, n_jobs=-1, verbosity=0
)
final_lgbm = LGBMRegressor(
    **study_lgbm.best_params,
    random_state=42, n_jobs=-1, verbose=-1
)

xgb_results  = evaluate(final_xgb,  X_train_sc, y_train, X_test_sc, y_test, 'Tuned XGBoost')
lgbm_results = evaluate(final_lgbm, X_train_sc, y_train, X_test_sc, y_test, 'Tuned LightGBM')

# Pick winner
winner_label   = 'XGBoost'  if xgb_results['r2'] >= lgbm_results['r2'] else 'LightGBM'
winner_model   = final_xgb  if xgb_results['r2'] >= lgbm_results['r2'] else final_lgbm
winner_results = xgb_results if xgb_results['r2'] >= lgbm_results['r2'] else lgbm_results

print(f'\n🏆 Winner: {winner_label} (R² = {winner_results["r2"]:.4f})')

# Improvement summary
improvement = winner_results['r2'] - base_results['r2']
mape_drop   = base_results['mape'] - winner_results['mape']
print(f'📈 R² improvement from baseline : +{improvement:.4f}')
print(f'📉 MAPE drop from baseline      : -{mape_drop:.2f}%')

In [ ]:
# ─── 11. Model Comparison Bar Chart ───────────────────────────────────────────
labels   = ['Baseline\nXGBoost', 'Tuned\nXGBoost', 'Tuned\nLightGBM']
r2_vals  = [base_results['r2'], xgb_results['r2'], lgbm_results['r2']]
mae_vals = [base_results['mae'], xgb_results['mae'], lgbm_results['mae']]
mape_vals= [base_results['mape'], xgb_results['mape'], lgbm_results['mape']]
colors   = ['#3a3d4a', ACCENT, ACCENT2]

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
fig.suptitle('Model Comparison — Baseline vs Tuned', fontsize=14, fontweight='bold')

for ax, vals, title, fmt, better in zip(
    axes,
    [r2_vals, mae_vals, mape_vals],
    ['R² Score (higher=better)', 'MAE $ (lower=better)', 'MAPE % (lower=better)'],
    ['.4f', '.2f', '.2f'],
    ['high', 'low', 'low']
):
    bars = ax.bar(labels, vals, color=colors, alpha=0.85, edgecolor='#0f1117', width=0.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
                f'{val:{fmt}}', ha='center', fontsize=9, fontweight='bold', color='#e8eaf0')
    ax.set_title(title)
    ax.grid(True, axis='y')
    ax.set_ylim(0, max(vals) * 1.18)

plt.tight_layout()
plt.show()

In [ ]:
# ─── 12. Feature Importance (Winner Model) ────────────────────────────────────
if hasattr(winner_model, 'feature_importances_'):
    importance = pd.Series(winner_model.feature_importances_, index=FEATURES).sort_values()

    top_n   = 15
    imp_top = importance.tail(top_n)

    fig, ax = plt.subplots(figsize=(10, 6))
    colors_bar = []
    for v in imp_top.values:
        if v == imp_top.max():          colors_bar.append(ACCENT)
        elif v >= imp_top.quantile(0.75): colors_bar.append(ACCENT2)
        else:                             colors_bar.append('#3a3d4a')

    bars = ax.barh(imp_top.index, imp_top.values,
                   color=colors_bar, edgecolor='#0f1117', height=0.65)

    for bar, val in zip(bars, imp_top.values):
        ax.text(val + imp_top.max()*0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=8.5, color='#9ca3af')

    ax.set_title(f'{winner_label} — Feature Importance (Top {top_n})',
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('Importance Score')
    ax.set_xlim(0, imp_top.max() * 1.2)
    ax.grid(True, axis='x')

    # Legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=ACCENT,  label='Top feature'),
        Patch(facecolor=ACCENT2, label='High importance'),
        Patch(facecolor='#3a3d4a', label='Standard'),
    ]
    ax.legend(handles=legend_elements, framealpha=0.2, loc='lower right')
    plt.tight_layout()
    plt.show()

    print(f'Top feature    : {importance.idxmax()} ({importance.max():.4f})')
    print(f'Bottom feature : {importance.idxmin()} ({importance.min():.4f})')

    engineered_importance = importance[engineered].sum()
    total_importance = importance.sum()
    print(f'\nEngineered features share: {engineered_importance/total_importance*100:.1f}% of total importance')

In [ ]:
# ─── 13. Prediction Error Analysis ────────────────────────────────────────────
y_pred    = winner_results['y_pred']
y_actual  = winner_results['y_actual']
residuals = y_actual - y_pred

fig = plt.figure(figsize=(15, 10))
fig.suptitle(f'Prediction Error Analysis — {winner_label}', fontsize=14, fontweight='bold')
gs  = gridspec.GridSpec(2, 3, figure=fig)

# 1. Actual vs Predicted
ax1 = fig.add_subplot(gs[0, 0:2])
sc  = ax1.scatter(y_actual, y_pred, alpha=0.4, s=12,
                  c=np.abs(residuals), cmap='RdYlGn_r', vmin=0, vmax=100)
lims = [min(y_actual.min(), y_pred.min()), max(y_actual.max(), y_pred.max())]
ax1.plot(lims, lims, color=WARN, linewidth=1.5, linestyle='--', label='Perfect prediction')
ax1.plot(lims, [l*1.2 for l in lims], color='#555', linewidth=1, linestyle=':', alpha=0.5, label='+20% band')
ax1.plot(lims, [l*0.8 for l in lims], color='#555', linewidth=1, linestyle=':', alpha=0.5, label='-20% band')
plt.colorbar(sc, ax=ax1, label='Abs Error ($)', shrink=0.8)
ax1.set_title(f'Actual vs Predicted  (R²={winner_results["r2"]:.4f})')
ax1.set_xlabel('Actual Price ($)')
ax1.set_ylabel('Predicted Price ($)')
ax1.legend(framealpha=0.2, fontsize=8)
ax1.grid(True)

# 2. Residuals distribution
ax2 = fig.add_subplot(gs[0, 2])
ax2.hist(residuals, bins=30, color=ACCENT2, alpha=0.85, edgecolor='#0f1117')
ax2.axvline(0, color=WARN, linestyle='--', lw=1.5)
ax2.axvline(residuals.mean(), color=ACCENT, linestyle='--', lw=1.2,
            label=f'Mean ${residuals.mean():.1f}')
ax2.set_title('Residuals Distribution')
ax2.set_xlabel('Residual ($)')
ax2.legend(framealpha=0.2)
ax2.grid(True)

# 3. Residuals vs Predicted
ax3 = fig.add_subplot(gs[1, 0])
ax3.scatter(y_pred, residuals, alpha=0.3, s=8, color=WARN)
ax3.axhline(0, color='#9ca3af', linestyle='--', lw=1.2)
ax3.set_title('Residuals vs Predicted')
ax3.set_xlabel('Predicted Price ($)')
ax3.set_ylabel('Residual ($)')
ax3.grid(True)

# 4. Error by price bucket
ax4 = fig.add_subplot(gs[1, 1])
buckets = pd.cut(y_actual, bins=5)
bucket_mae = pd.Series(np.abs(residuals)).groupby(buckets).mean()
ax4.bar(range(len(bucket_mae)), bucket_mae.values, color=PURPLE, alpha=0.8, edgecolor='#0f1117')
ax4.set_xticks(range(len(bucket_mae)))
ax4.set_xticklabels([str(b) for b in bucket_mae.index], rotation=30, fontsize=7)
ax4.set_title('MAE by Price Range')
ax4.set_xlabel('Price Bucket ($)')
ax4.set_ylabel('MAE ($)')
ax4.grid(True, axis='y')

# 5. % within tolerance bands
ax5 = fig.add_subplot(gs[1, 2])
thresholds = [10, 20, 30, 50, 75, 100]
pcts = [(np.abs(residuals) < t).mean() * 100 for t in thresholds]
bars_t = ax5.bar([f'${t}' for t in thresholds], pcts,
                  color=[ACCENT if p >= 70 else ACCENT2 if p >= 50 else WARN for p in pcts],
                  alpha=0.85, edgecolor='#0f1117')
for bar, pct in zip(bars_t, pcts):
    ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{pct:.1f}%', ha='center', fontsize=8, color='#e8eaf0')
ax5.set_title('% Predictions Within Error Band')
ax5.set_xlabel('Error Tolerance')
ax5.set_ylabel('% Predictions')
ax5.set_ylim(0, 115)
ax5.grid(True, axis='y')

plt.tight_layout()
plt.show()

print(f'Mean residual              : ${residuals.mean():.2f}')
print(f'Std residual               : ${residuals.std():.2f}')
print(f'% predictions within $20   : {(np.abs(residuals) < 20).mean()*100:.1f}%')
print(f'% predictions within $50   : {(np.abs(residuals) < 50).mean()*100:.1f}%')
print(f'% predictions within $100  : {(np.abs(residuals) < 100).mean()*100:.1f}%')

In [ ]:
# ─── 14. Optuna Optimization History ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Optuna Optimization History', fontsize=14, fontweight='bold')

for ax, study, label, color in [
    (axes[0], study_xgb,  'XGBoost',  ACCENT),
    (axes[1], study_lgbm, 'LightGBM', ACCENT2)
]:
    trials = [t.value for t in study.trials if t.value is not None]
    best_so_far = np.maximum.accumulate(trials)
    ax.plot(trials,       alpha=0.4, color=color,  linewidth=0.8, label='Trial R²')
    ax.plot(best_so_far,  alpha=0.9, color=WARN,   linewidth=2,   label='Best so far')
    ax.set_title(f'{label} — CV R² over trials')
    ax.set_xlabel('Trial')
    ax.set_ylabel('CV R²')
    ax.legend(framealpha=0.2)
    ax.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# ─── 15. Final Summary & Save Artifacts ───────────────────────────────────────
import os
os.makedirs('../models', exist_ok=True)

joblib.dump(winner_model, '../models/price_model.joblib')
joblib.dump(scaler,       '../models/scaler.joblib')

meta = {
    'model_type'   : winner_label,
    'features'     : FEATURES,
    'target'       : 'log1p(Historical_Cost_of_Ride)',
    'inverse'      : 'expm1',
    'metrics': {
        'MAE'    : round(winner_results['mae'],  2),
        'RMSE'   : round(winner_results['rmse'], 2),
        'R2'     : round(winner_results['r2'],   4),
        'MAPE'   : round(winner_results['mape'], 2),
        'CV_R2'  : round(winner_results['cv_mean'], 4),
        'CV_std' : round(winner_results['cv_std'],  4),
    },
    'baseline_r2'  : round(base_results['r2'],   4),
    'improvement'  : round(winner_results['r2'] - base_results['r2'], 4),
    'best_params'  : study_xgb.best_params if winner_label == 'XGBoost' else study_lgbm.best_params
}
with open('../models/model_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('Artifacts saved ✅')
print('  ../models/price_model.joblib')
print('  ../models/scaler.joblib')
print('  ../models/model_meta.json')

print(f'''
╔══════════════════════════════════════════════════╗
║           FINAL MODEL REPORT                     ║
╠══════════════════════════════════════════════════╣
║  Model       : {winner_label:<33}║
║  MAE         : ${winner_results['mae']:<32.2f}║
║  RMSE        : ${winner_results['rmse']:<32.2f}║
║  R²          : {winner_results['r2']:<33.4f}║
║  MAPE        : {winner_results['mape']:<32.2f}%║
║  CV R²       : {winner_results['cv_mean']:.4f} ± {winner_results['cv_std']:.4f}              ║
╠══════════════════════════════════════════════════╣
║  Baseline R² : {base_results['r2']:<33.4f}║
║  Improvement : +{winner_results['r2'] - base_results['r2']:<32.4f}║
╚══════════════════════════════════════════════════╝
''')